# Part 3. Fox Glacier B08 autoRIFT

This notebook reads the two B08 GeoTIFFs downloaded in Part 2, runs autoRIFT feature tracking, and saves the final PNG/NPZ results in the current folder.


## Colab setup

If you are using Google Colab, run the next cell first. autoRIFT and GDAL are conda-forge packages, so this setup uses `condacolab`/`mamba` instead of only pip. After `condacolab` restarts the runtime, run the cell again and then continue.


In [ ]:
# Colab setup for Part 3: autoRIFT + GDAL/raster stack.
from __future__ import annotations

import importlib.util
import shutil
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
NEEDED = ["autoRIFT", "geogrid", "rasterio", "scipy", "shapely", "pyproj", "numpy", "matplotlib"]
missing = [m for m in NEEDED if importlib.util.find_spec(m) is None]

if IN_COLAB and missing:
    if shutil.which("mamba") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "condacolab"], check=True)
        import condacolab
        condacolab.install()
        raise SystemExit("Colab runtime restarted. Run this setup cell again, then continue.")
    subprocess.run(
        [
            "mamba",
            "install",
            "-y",
            "-c",
            "conda-forge",
            "autorift",
            "gdal",
            "rasterio",
            "scipy",
            "shapely",
            "pyproj",
            "numpy",
            "matplotlib",
        ],
        check=True,
    )
else:
    print("Part 3 environment looks ready, or this is not Colab.")


In [ ]:
from __future__ import annotations

import argparse
import re
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.colors import Normalize
from osgeo import gdal
from rasterio.plot import plotting_extent

gdal.UseExceptions()

from autoRIFT import autoRIFT
from geogrid import GeogridOptical


In [ ]:
ROOT = Path.cwd().resolve()
DEFAULT_REF = ROOT / "B08_fox_2026-03-01.tif"
DEFAULT_SEC = ROOT / "B08_fox_2026-03-21.tif"
DEFAULT_OUT = ROOT
DEFAULT_GLACIER_BOUNDARY = ROOT / "fox_glacier_from_osm.geojson"

DATE_RE = re.compile(r"B08_fox_(\d{4}-\d{2}-\d{2})")

# Larger GRID_STEP / MARGIN gives fewer arrows and faster runtime.
WALLIS_WIDTH = 7
SEARCH_LIMIT = 72.0
GRID_STEP = 12
MARGIN = 48


In [ ]:
def _parse_date(path: Path) -> date | None:
    m = DATE_RE.search(path.name)
    return date.fromisoformat(m.group(1)) if m else None


def load_overlap_pair(
    ref_path: Path, sec_path: Path
) -> tuple[np.ndarray, np.ndarray, tuple[float, ...], int, int]:
    gg = GeogridOptical()
    x1a, y1a, xsize, ysize, x2a, y2a, _xs2, _ys2, trans = gg.coregister(str(ref_path), str(sec_path))
    ds1 = gdal.Open(str(ref_path), gdal.GA_ReadOnly)
    ds2 = gdal.Open(str(sec_path), gdal.GA_ReadOnly)
    if ds1 is None or ds2 is None:
        raise FileNotFoundError("GDAL could not open inputs")
    I1 = ds1.ReadAsArray(int(x1a), int(y1a), int(xsize), int(ysize)).astype(np.float32)
    I2 = ds2.ReadAsArray(int(x2a), int(y2a), int(xsize), int(ysize)).astype(np.float32)
    ds1 = ds2 = None
    I1[I1 <= 0] = 0.0
    I2[I2 <= 0] = 0.0
    return I1, I2, trans, int(x1a), int(y1a)


def build_pixel_grid_1based(
    height: int, width: int, *, margin: int, step: int, chop_factor: int
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    xs = np.arange(margin, width - margin + 1, step, dtype=np.int32)
    ys = np.arange(margin, height - margin + 1, step, dtype=np.int32)
    nc = (xs.size // chop_factor) * chop_factor
    nr = (ys.size // chop_factor) * chop_factor
    if nc < 4 or nr < 4:
        raise ValueError("Grid too small; reduce margin or step")
    xs, ys = xs[:nc], ys[:nr]
    xg = np.dot(np.ones((nr, 1), np.float32), xs.reshape(1, nc).astype(np.float32))
    yg = np.dot(ys.reshape(nr, 1).astype(np.float32), np.ones((1, nc), np.float32))
    return xg, yg, np.zeros((nr, nc), dtype=bool)




In [ ]:
def run_autorift_optical(
    I1: np.ndarray,
    I2: np.ndarray,
    x_grid: np.ndarray,
    y_grid: np.ndarray,
    no_data_mask: np.ndarray,
    *,
    wallis_width: int,
    search_limit: float,
    threads: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    obj = autoRIFT()
    obj.I1 = I1
    obj.I2 = I2
    obj.xGrid = x_grid.astype(np.float32)
    obj.yGrid = y_grid.astype(np.float32)
    obj.WallisFilterWidth = wallis_width
    obj.MultiThread = threads
    obj.ChipSizeMaxX = 32
    obj.ChipSizeMinX = 16
    obj.ChipSize0X = 16
    obj.GridSpacingX = 16

    z = np.zeros_like(no_data_mask, dtype=np.float32)
    obj.Dx0 = z
    obj.Dy0 = z.copy()
    obj.SearchLimitX = float(search_limit) * np.logical_not(no_data_mask)
    obj.SearchLimitY = float(search_limit) * np.logical_not(no_data_mask)

    obj.xGrid[no_data_mask] = 0.0
    obj.yGrid[no_data_mask] = 0.0
    obj.Dx0[no_data_mask] = 0.0
    obj.Dy0[no_data_mask] = 0.0
    obj.SearchLimitX[no_data_mask] = 0.0
    obj.SearchLimitY[no_data_mask] = 0.0

    print("  Wallis preprocess...")
    obj.preprocess_filt_wal()

    for name, arr in (("I1", obj.I1), ("I2", obj.I2)):
        if obj.zeroMask is not None:
            valid = np.isfinite(arr)
            s = float(np.std(arr[valid]) * np.sqrt(arr[valid].size / (arr[valid].size - 1.0)))
            m = float(np.mean(arr[valid]))
        else:
            s = float(np.std(arr) * np.sqrt(arr.size / (arr.size - 1.0)))
            m = float(np.mean(arr))
        scaled = (arr - (m - 3 * s)) / (6 * s) * 255.0
        scaled_u8 = np.round(np.clip(scaled, 0, 255)).astype(np.uint8)
        if name == "I1":
            obj.I1 = scaled_u8
        else:
            obj.I2 = scaled_u8

    if obj.zeroMask is not None:
        obj.I1[obj.zeroMask] = 0
        obj.I2[obj.zeroMask] = 0
        obj.zeroMask = None

    obj.OverSampleRatio = 64
    print("  runAutorift()...")
    obj.runAutorift()
    return obj.Dx, obj.Dy, obj.xGrid, obj.yGrid


def offsets_to_m_per_day(
    dx_px: np.ndarray, dy_px: np.ndarray, gdal_gt: tuple[float, ...], dt_days: float
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    gt0, gt1, gt2, gt3, gt4, gt5 = gdal_gt[:6]
    east_m = dx_px * gt1 + dy_px * gt2
    north_m = dx_px * gt4 + dy_px * gt5
    vx = east_m / dt_days
    vy = north_m / dt_days
    # Fox: naive E–N can look SW vs expected NW; use clockwise 90° in map plane: (vx', vy') = (vy, -vx)
    vx, vy = vy, -vx
    spd = np.hypot(vx, vy)
    return vx, vy, spd



In [ ]:
def _projected_glacier_union(geojson_path: Path, ref_path: Path):
    """GeoJSON (CRS84) glacier union → same CRS as ``ref_path`` raster; ``None`` if deps/path missing."""
    if not geojson_path.is_file():
        return None
    try:
        import json
        from shapely.geometry import shape
        from shapely.ops import transform, unary_union
        import pyproj
    except ImportError:
        return None
    with rasterio.open(ref_path) as src:
        crs_dst = src.crs
    if crs_dst is None:
        return None
    g = json.loads(geojson_path.read_text(encoding="utf-8"))
    feats = g.get("features", [])
    if not feats:
        return None
    geoms = []
    for f in feats:
        geom = shape(f["geometry"])
        if geom.is_empty:
            continue
        geoms.append(geom)
    if not geoms:
        return None
    lonlat = unary_union(geoms)
    tfm = pyproj.Transformer.from_crs("EPSG:4326", crs_dst, always_xy=True)

    def _proj(x, y, z=None):
        return tfm.transform(x, y)

    return transform(_proj, lonlat)


def _gaussian_kde_1d(samples: np.ndarray, bw_factor: float):
    """``gaussian_kde`` with ``bw_method=bw_factor`` (scalar = bandwidth multiplier)."""
    from scipy.stats import gaussian_kde

    s = np.asarray(samples, dtype=float)
    s = s[np.isfinite(s)]
    if s.size < 2:
        raise ValueError("KDE needs at least 2 samples")
    bf = float(bw_factor) if bw_factor > 0 else 1.0
    try:
        return gaussian_kde(s, bw_method=bf)
    except TypeError:
        return gaussian_kde(s)


def plot_kde_glacier_buffer(
    *,
    east: np.ndarray,
    north: np.ndarray,
    spd: np.ndarray,
    ok: np.ndarray,
    tag: str,
    out_path: Path,
    dpi: int,
    ref_path: Path,
    glacier_boundary: Path | None,
    buffer_m: float,
    tail_frac: float,
    speed_xmax: float,
    kde_color: str = "darkred",
    kde_bw_factor: float = 2.0,
) -> None:
    """KDE figure: histograms + two KDEs on equal-count slow/fast tails inside glacier ``+ buffer_m``.

    ``tail_frac``: each tail size ≈ ``min(max(10, tail_frac*n), n//2)`` → changes which chips are compared.
    ``speed_xmax``: x-axis ``[0, xmax]`` for hist + KDE grid.
    ``kde_bw_factor``: **slow tail only** (wider → lower peak near 0); fast tail is always default ``gaussian_kde``.
    ``kde_color``: hist + both KDE lines. Legend shows a single "KDE" entry.
    """
    try:
        from scipy.stats import gaussian_kde
    except ImportError:
        print("  skip KDE")
        return
    try:
        from shapely import contains_xy
    except ImportError:
        print("  skip KDE")
        return

    bpath = glacier_boundary if glacier_boundary is not None else DEFAULT_GLACIER_BOUNDARY
    glac = _projected_glacier_union(bpath, ref_path) if bpath.is_file() else None
    if glac is None:
        print("  skip KDE (boundary)")
        return

    try:
        buf = glac.buffer(float(buffer_m))
    except Exception as e:
        print(f"  skip KDE (buffer failed: {e})")
        return

    e = np.asarray(east, dtype=float).ravel()
    n = np.asarray(north, dtype=float).ravel()
    s = np.asarray(spd, dtype=float).ravel()
    o = np.asarray(ok, dtype=bool).ravel()
    m = o & np.isfinite(s) & (s < 1e6) & contains_xy(buf, e, n)
    if int(np.count_nonzero(m)) < 40:
        print(f"  skip KDE (only {int(np.count_nonzero(m))} chips in glacier+{buffer_m:.0f} m buffer)")
        return

    s2 = s[m]
    order = np.argsort(s2)
    nbuf = int(s2.size)
    k = min(max(10, int(tail_frac * nbuf)), nbuf // 2)
    if k < 10:
        print("  skip KDE (buffer sample too small for tail split)")
        return

    sta = order[:k]
    mov = order[-k:]
    sta_s = s2[sta]
    mov_s = s2[mov]

    xs = np.linspace(0.0, float(speed_xmax), 320)
    kde_lo = _gaussian_kde_1d(sta_s, kde_bw_factor)
    kde_hi = gaussian_kde(mov_s)

    c = kde_color
    fig, ax0 = plt.subplots(figsize=(8.5, 5.2), constrained_layout=True)
    nb = min(40, max(12, k // 2))
    ax0.hist(
        sta_s,
        bins=nb,
        range=(0.0, speed_xmax),
        density=True,
        alpha=0.38,
        color=c,
    )
    ax0.hist(
        mov_s,
        bins=nb,
        range=(0.0, speed_xmax),
        density=True,
        alpha=0.38,
        color=c,
    )
    ax0.plot(xs, kde_lo(xs), color=c, lw=2.2, ls="-", label="KDE")
    ax0.plot(xs, kde_hi(xs), color=c, lw=2.2, ls="-")
    ax0.set_xlim(0.0, float(speed_xmax))
    ax0.set_xlabel("Demeaned |v| (m/day)")
    ax0.set_ylabel("Density")
    ax0.set_title(tag, fontsize=10)
    ax0.legend(loc="upper right", fontsize=9)
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"  wrote {out_path.name}")


In [ ]:
def plot_quiver_scatter(
    *,
    tag: str,
    out_dir: Path,
    gdal_gt: tuple[float, ...],
    xg: np.ndarray,
    yg: np.ndarray,
    vx: np.ndarray,
    vy: np.ndarray,
    spd: np.ndarray,
    overlap_h: int,
    overlap_w: int,
    exaggerate: float,
    dpi: int,
    ref_path: Path,
    xoff: int,
    yoff: int,
    glacier_boundary: Path | None = None,
    kde_buffer_m: float = 400.0,
    kde_tail_frac: float = 0.25,
    kde_speed_xmax: float = 1.5,
    kde_color: str = "darkred",
    kde_bw_factor: float = 2.0,
) -> None:
    """Save NPZ + quiver + speed scatter + KDE. ``exaggerate``/``dpi`` only affect PNGs.

    Demean: subtracts ``nanmedian`` of vx, vy → changes ``spd``, scatter, KDE inputs.
    ``kde_buffer_m``: KDE uses chips inside buffered glacier polygon only.
    ``kde_tail_frac``, ``kde_speed_xmax``, ``kde_color``, ``kde_bw_factor``: see ``plot_kde_glacier_buffer``.
    """
    gt0, gt1, gt2, gt3, gt4, gt5 = gdal_gt[:6]
    col0 = np.asarray(xg, dtype=float) - 1.0
    row0 = np.asarray(yg, dtype=float) - 1.0
    east = gt0 + col0 * gt1 + row0 * gt2
    north = gt3 + col0 * gt4 + row0 * gt5

    vx_raw = np.asarray(vx, dtype=float)
    vy_raw = np.asarray(vy, dtype=float)

    vx = vx_raw - float(np.nanmedian(vx_raw))
    vy = vy_raw - float(np.nanmedian(vy_raw))
    spd = np.hypot(vx, vy)

    ok = np.isfinite(spd) & (spd < 1e6)
    vmax = float(np.nanpercentile(spd[ok], 96)) if np.any(ok) else 1e-3
    vmax = max(vmax, 1e-4)
    suf = "_demeaned"

    npz_path = out_dir / f"autorift_scatter_demeaned_{tag}.npz"
    np.savez_compressed(
        npz_path,
        easting_m=east[ok].astype(np.float64),
        northing_m=north[ok].astype(np.float64),
        speed_m_per_day=spd[ok].astype(np.float32),
        vx_m_per_day_demeaned=vx[ok].astype(np.float32),
        vy_m_per_day_demeaned=vy[ok].astype(np.float32),
    )
    print(f"  wrote {npz_path.name}")

    fig, ax = plt.subplots(figsize=(11, 9))
    win = rasterio.windows.Window(xoff, yoff, overlap_w, overlap_h)
    with rasterio.open(ref_path) as src:
        arr = src.read(1, window=win).astype(np.float32)
        tr = src.window_transform(win)
    v = arr[np.isfinite(arr) & (arr > 0)]
    lo, hi = np.percentile(v, (2, 98)) if v.size else (0.0, 1.0)
    hi = max(hi, lo + 1e-6)
    bg = np.clip((arr - lo) / (hi - lo), 0, 1)
    extent = plotting_extent(bg, tr)
    ax.imshow(bg, extent=extent, origin="upper", cmap="gray", aspect="equal", alpha=0.9)
    ax.quiver(
        east,
        north,
        np.where(ok, vx * exaggerate, np.nan),
        np.where(ok, vy * exaggerate, np.nan),
        angles="xy",
        scale_units="xy",
        scale=1.0,
        width=0.0025,
        color="cyan",
        headwidth=4,
    )
    ax.set_title(
        f"autoRIFT POT (m/day) demeaned | x{exaggerate:g} exaggeration\n{tag}",
        fontsize=10,
    )
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    fig.tight_layout()
    pq = out_dir / f"autorift_pot_quiver{suf}_{tag}.png"
    fig.savefig(pq, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"  wrote {pq.name}")

    fig2, ax2 = plt.subplots(figsize=(10, 8))
    sc = ax2.scatter(east[ok], north[ok], c=spd[ok], s=40, cmap="viridis", norm=Normalize(0.0, vmax))
    plt.colorbar(sc, ax=ax2, label="speed (m/day)")
    ax2.set_aspect("equal")
    ax2.set_title(f"|velocity| (m/day) demeaned\n{tag}", fontsize=10)
    ax2.set_xlabel("Easting (m)")
    ax2.set_ylabel("Northing (m)")
    fig2.tight_layout()
    ps = out_dir / f"autorift_pot_speed_scatter{suf}_{tag}.png"
    fig2.savefig(ps, dpi=dpi, bbox_inches="tight")
    plt.close(fig2)
    print(f"  wrote {ps.name}")

    pk = out_dir / f"autorift_pot_kde{suf}_{tag}.png"
    plot_kde_glacier_buffer(
        east=east,
        north=north,
        spd=spd,
        ok=ok,
        tag=tag,
        out_path=pk,
        dpi=dpi,
        ref_path=ref_path,
        glacier_boundary=glacier_boundary,
        buffer_m=kde_buffer_m,
        tail_frac=kde_tail_frac,
        speed_xmax=kde_speed_xmax,
        kde_color=kde_color,
        kde_bw_factor=kde_bw_factor,
    )


Defaults read the two B08 GeoTIFFs and boundary from the current lab folder. Edit `threads` or plotting parameters if needed.


In [ ]:
ref = DEFAULT_REF
sec = DEFAULT_SEC
out_dir = DEFAULT_OUT
threads = 4
exaggerate = 120.0
dpi = 160
glacier_boundary = DEFAULT_GLACIER_BOUNDARY
kde_buffer_m = 400.0
kde_speed_xmax = 1.5
kde_tail_frac = 0.25
kde_color = "darkred"

kde_bw_factor = 2.0


In [ ]:
ref, sec = ref.resolve(), sec.resolve()
out_dir = out_dir.resolve()
out_dir.mkdir(parents=True, exist_ok=True)
if not ref.is_file() or not sec.is_file():
    raise SystemExit(f"Missing inputs: {ref} or {sec}")

d0, d1 = _parse_date(ref), _parse_date(sec)
if d0 is None or d1 is None:
    tag = f"{ref.stem}__{sec.stem}"
    dt_days = 20.0
    print("Warning: could not parse dates from filenames; using dt_days=20")
else:
    tag = f"{d0.isoformat()}__{d1.isoformat()}"
    dt_days = float((d1 - d0).days)
    if dt_days <= 0:
        raise SystemExit("Secondary must be later than reference")

I1, I2, gdal_gt, xoff, yoff = load_overlap_pair(ref, sec)
m, n = I1.shape
xg, yg, nodata = build_pixel_grid_1based(m, n, margin=MARGIN, step=GRID_STEP, chop_factor=2)
print(f"overlap {n}x{m} | grid {xg.shape[0]}x{xg.shape[1]} | W={WALLIS_WIDTH} SL={SEARCH_LIMIT}")

dx, dy, xg2, yg2 = run_autorift_optical(
    I1, I2, xg, yg, nodata, wallis_width=WALLIS_WIDTH, search_limit=SEARCH_LIMIT, threads=threads
)
finite = np.isfinite(dx) & np.isfinite(dy)
print(
    f"  finite Dx,Dy {finite.sum()}/{finite.size} ({100 * finite.mean():.1f}%) — "
    "NaN = no arrow; larger GRID_STEP / MARGIN = fewer chip sites overall"
)
vx, vy, spd = offsets_to_m_per_day(dx, dy, gdal_gt, dt_days)

plot_quiver_scatter(
    tag=tag,
    out_dir=out_dir,
    gdal_gt=gdal_gt,
    xg=xg2,
    yg=yg2,
    vx=vx,
    vy=vy,
    spd=spd,
    overlap_h=m,
    overlap_w=n,
    exaggerate=exaggerate,
    dpi=dpi,
    ref_path=ref,
    xoff=xoff,
    yoff=yoff,
    glacier_boundary=glacier_boundary,
    kde_buffer_m=kde_buffer_m,
    kde_tail_frac=kde_tail_frac,
    kde_speed_xmax=kde_speed_xmax,
    kde_color=kde_color,
    kde_bw_factor=kde_bw_factor,
)
print("Done.")
